# <center>ML-7. Прогнозирование биологического ответа (HW-3)

Данные представлены в формате CSV.  Каждая строка представляет молекулу. 

* Первый столбец *Activity* содержит экспериментальные данные, описывающие фактический биологический ответ [0, 1]; 
* Остальные столбцы D1-D1776 представляют собой молекулярные **дескрипторы** — это вычисляемые свойства, которые могут фиксировать некоторые характеристики молекулы, например размер, форму или состав элементов.

Необходимо обучить две модели: `логистическую регрессию` и `случайный лес`. Далее нужно сделать подбор гиперпараметров с помощью базовых и продвинутых методов оптимизации. Важно использовать все четыре метода (`GridSeachCV`, `RandomizedSearchCV`, `Hyperopt`, `Optuna`) хотя бы по разу, максимальное количество итераций не должно превышать 50.

**КРИТЕРИИ ОЦЕНКИ**

Балл	Критерий
* 0	Задание не выполнено
* 1	Обучено две модели; гипепараметры подобраны при помощи одного метода
* 2	Обучено две модели; гипепараметры подобраны при помощи двух методов
* 3	Обучено две модели; гипепараметры подобраны при помощи трёх методов
* 4	Обучено две модели; гипепараметры подобраны при помощи четырёх методов
* 5	Обучено две модели; гипепараметры подобраны при помощи четырёх методов; использована кросс-валидация

In [16]:
#импорт библиотек
import numpy as np #для матричных вычислений
import pandas as pd #для анализа и предобработки данных
import matplotlib.pyplot as plt #для визуализации
import seaborn as sns #для визуализации

from sklearn import linear_model #линейные моделиё
from sklearn import tree #деревья решений
from sklearn import ensemble #ансамбли
from sklearn import metrics #метрики
from sklearn import preprocessing #предобработка
from sklearn.model_selection import train_test_split #сплитование выборки
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
import hyperopt
from hyperopt import hp, fmin, tpe, Trials
import optuna


%matplotlib inline

In [2]:
data = pd.read_csv('_train_sem09.csv')
data.head()

,Activity,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,D1767,D1768,D1769,D1770,D1771,D1772,D1773,D1774,D1775,D1776
0,1,0.000000,0.497009,0.10,0.0,0.132956,0.678031,0.273166,0.585445,0.743663,...,0,0,0,0,0,0,0,0,0,0
1,1,0.366667,0.606291,0.05,0.0,0.111209,0.803455,0.106105,0.411754,0.836582,...,1,1,1,1,0,1,0,0,1,0
2,1,0.033300,0.480124,0.00,0.0,0.209791,0.610350,0.356453,0.517720,0.679051,...,0,0,0,0,0,0,0,0,0,0
3,1,0.000000,0.538825,0.00,0.5,0.196344,0.724230,0.235606,0.288764,0.805110,...,0,0,0,0,0,0,0,0,0,0
4,0,0.100000,0.517794,0.00,0.0,0.494734,0.781422,0.154361,0.303809,0.812646,...,0,0,0,0,0,0,0,0,0,0


Поскольку предобработка не требуется, а данные уже закодированы и нормализованы, то создаем матрицу наблюдений $X$ и вектор ответов $y$

In [4]:
X = data.drop(['Activity'], axis=1)
y = data['Activity']

In [5]:
#Разделяем выборку на тренировочную и тестовую в соотношении 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state = 1, test_size = 0.2)

Зафиксируем метрики, которые были получены со значениями гиперпараметров, установленных по умолчанию:

In [9]:
#Создаем объект класса логистическая регрессия
log_reg = linear_model.LogisticRegression(max_iter = 1000, random_state=42)

log_reg.fit(X_train, y_train)
y_test_pred = log_reg.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

f1_score на тестовом наборе: 0.79


c:\Users\Zylno\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
#Создаем объект класса случайный лес
rf = ensemble.RandomForestClassifier(n_estimators=500, random_state=42)

rf.fit(X_train, y_train)
y_test_pred = rf.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

f1_score на тестовом наборе: 0.83


### <center> GridSearchCV

In [13]:
#Производим подбор гиперпараметров для модели логистической регрессии
param_grid_lr = [
               {'penalty': ['l1'], 
                'C': [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 1], 
                'solver': ['liblinear', 'saga']},
               
                {'penalty': ['l2'], 
                 'C': [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 1], 
                 'solver': ['lbfgs', 'sag']}
]
grid_search_lr = GridSearchCV(
    estimator=linear_model.LogisticRegression(
        random_state=42, #генератор случайных чисел
        max_iter=1000 #количество итераций на сходимость
    ), 
    param_grid=param_grid_lr, 
    cv=5, 
    n_jobs = -1
)  
%time grid_search_lr.fit(X_train, y_train) 
y_test_pred = grid_search_lr.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(grid_search_lr.best_params_))

CPU times: total: 10.3 s
Wall time: 1min 42s
f1_score на тестовом наборе: 0.79
Наилучшие значения гиперпараметров: {'C': 0.1, 'penalty': 'l2', 'solver': 'sag'}


>Значение метрики не изменилось

In [15]:
#Производим подбор гиперпараметров для модели случайного леса
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_search_rf = GridSearchCV(
    estimator=ensemble.RandomForestClassifier(
        random_state=42, #генератор случайных чисел
    ), 
    param_grid=param_grid_rf, 
    cv=5, 
    n_jobs = -1
)  
%time grid_search_rf.fit(X_train, y_train) 
y_test_pred = grid_search_rf.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(grid_search_rf.best_params_))

CPU times: total: 2.52 s
Wall time: 18 s
f1_score на тестовом наборе: 0.83
Наилучшие значения гиперпараметров: {'max_depth': 20, 'min_samples_split': 10, 'n_estimators': 200}


>Значение метрики так же не изменилось

### <center> RandomizedSearchCV

In [22]:
#Производим подбор гиперпараметров для модели логистической регрессии
param_distributions_lr = [{'penalty': ['l1'] ,
              'solver': ['liblinear', 'saga'],
               'C': list(np.linspace(0.01, 1, 10, dtype=float))},
                          
                {'penalty': ['l2'],  
                 'solver': ['lbfgs', 'sag'],
                 'C': list(np.linspace(0.01, 1, 10, dtype=float))}
]

random_search_lr = RandomizedSearchCV(
    estimator=linear_model.LogisticRegression(random_state=42, max_iter=1000), 
    param_distributions=param_distributions_lr, 
    cv=5, 
    n_iter = 10, 
    n_jobs = -1
)  
%time random_search_lr.fit(X_train, y_train) 
y_test_pred = random_search_lr.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(random_search_lr.best_params_))

CPU times: total: 16 s
Wall time: 58.8 s
f1_score на тестовом наборе: 0.78
Наилучшие значения гиперпараметров: {'solver': 'sag', 'penalty': 'l2', 'C': 0.34}


>Значение метрики улучшить не удалось

In [29]:
#Производим подбор гиперпараметров для модели случайного леса
param_distributions_rf = {'n_estimators': list(range(50, 200, 50)),
              'min_samples_leaf': [2, 5, 10],
              'max_depth': list(np.linspace(10, 20, 5, dtype=int))
              }
            
random_search_rf = RandomizedSearchCV(
    estimator=ensemble.RandomForestClassifier(random_state=42), 
    param_distributions=param_distributions_rf, 
    cv=5,
    n_iter = 20, 
    n_jobs = -1
)  
%time random_search_rf.fit(X_train, y_train) 
y_test_pred = random_search_rf.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(random_search_rf.best_params_))

CPU times: total: 750 ms
Wall time: 9.65 s
f1_score на тестовом наборе: 0.80
Наилучшие значения гиперпараметров: {'n_estimators': 50, 'min_samples_leaf': 2, 'max_depth': 17}


>Значение метрики улучшить не удалось

### <center> Hyperopt

In [63]:
# зададим пространство поиска гиперпараметров модели линейной регрессии
space_lr = {
    'penalty': hp.choice('penalty', ['l1', 'l2']),
    'C': hp.uniform('C', 0.1, 1),
    'solver': hp.choice('solver', ['liblinear'])
}

random_state = 42
def hyperopt_lr(params, cv=5, X=X_train, y=y_train, random_state=random_state):
    # функция получает комбинацию гиперпараметров в "params"
    params = {'penalty': str(params['penalty']), 
              'C': float(params['C']), 
             'solver': str(params['solver'])
              }
    
    model = linear_model.LogisticRegression(**params, random_state=random_state)

    # обучаем модель
    model.fit(X, y)
    score = metrics.f1_score(y, model.predict(X))
    
    return -score

# начинаем подбор гиперпараметров

trials = Trials() # используется для логирования результатов

best=fmin(hyperopt_lr, # наша функция 
          space=space_lr, # пространство гиперпараметров
          max_evals=30, # максимальное количество итераций
          trials=trials, # логирование результатов
          rstate=np.random.default_rng(random_state)# фиксируем для повторяемости результата
         )
print("Наилучшие значения гиперпараметров {}".format(best))

TPE is being used as the default algorithm.


100%|██████████| 30/30 [00:07<00:00,  3.91trial/s, best loss: -0.8856968215158925]
Наилучшие значения гиперпараметров {'C': 0.997288488910095, 'penalty': 1, 'solver': 0}


In [54]:
# Преобразуем лучшие параметры
best_params = {
    'penalty': ['l1', 'l2'][best['penalty']],
    'C': best['C'],
    'solver': ['liblinear'][best['solver']]
}

# рассчитаем точность для тестовой выборки
model_lr = linear_model.LogisticRegression(
    random_state=random_state, 
    penalty=best_params['penalty'],
    C=best_params['C'],
    solver=best_params['solver']
)
model_lr.fit(X_train, y_train)

y_test_pred = model_lr.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

f1_score на тестовом наборе: 0.78


>Значение метрики улучшить не удалось

In [55]:
# зададим пространство поиска гиперпараметров модели случайного леса
space_rf={'n_estimators': hp.quniform('n_estimators', 100, 200, 10),
       'max_depth' : hp.quniform('max_depth', 15, 40, 1),
       'min_samples_leaf': hp.quniform('min_samples_leaf', 3, 10, 1)
      }

random_state = 42
def hyperopt_rf(params, cv=5, X=X_train, y=y_train, random_state=random_state):
    params = {'n_estimators': int(params['n_estimators']), 
              'max_depth': int(params['max_depth']), 
             'min_samples_leaf': int(params['min_samples_leaf'])
              }
  
    # используем эту комбинацию для построения модели
    model = ensemble.RandomForestClassifier(**params, random_state=random_state)

    # обучаем модель
    model.fit(X, y)
    score = metrics.f1_score(y, model.predict(X))

    return -score

trials = Trials() # используется для логирования результатов

best=fmin(hyperopt_rf, # наша функция 
          space=space_rf, # пространство гиперпараметров
          algo=tpe.suggest, # алгоритм оптимизации, установлен по умолчанию, задавать необязательно
          max_evals=20, # максимальное количество итераций
          trials=trials, # логирование результатов
          rstate=np.random.default_rng(random_state)# фиксируем для повторяемости результата
         )
print("Наилучшие значения гиперпараметров {}".format(best))

100%|██████████| 20/20 [00:22<00:00,  1.11s/trial, best loss: -0.9748774509803921]
Наилучшие значения гиперпараметров {'max_depth': 22.0, 'min_samples_leaf': 3.0, 'n_estimators': 100.0}


In [56]:
# рассчитаем точность для тестовой выборки
model_rf = ensemble.RandomForestClassifier(
    random_state=random_state, 
    n_estimators=int(best['n_estimators']),
    max_depth=int(best['max_depth']),
    min_samples_leaf=int(best['min_samples_leaf'])
)
model_rf.fit(X_train, y_train)

y_test_pred = model_rf.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

f1_score на тестовом наборе: 0.83


>Значение метрики улучшить не удалось

### <center> Optuna

In [68]:
def optuna_lr(trial):
  # задаем пространства поиска гиперпараметров
  penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
  C = trial.suggest_float('C', 0.1, 1)
  solver = trial.suggest_categorical('solver', ['liblinear', 'saga', 'sag'])

  # создаем модель
  model = linear_model.LogisticRegression(penalty=penalty,
                                          C=C,
                                          solver=solver,
                                          random_state=random_state)
  # Исключение некорректных комбинаций
  if (solver in ['sag', 'saga'] and penalty == 'l1'):
      return np.inf
  # обучаем модель
  model.fit(X_train, y_train)
  score = metrics.f1_score(y_train, model.predict(X_train))

  return score

# cоздаем объект исследования
study_lr = optuna.create_study(study_name="LogisticRegression", direction="maximize")
# ищем лучшую комбинацию гиперпараметров n_trials раз
study_lr.optimize(optuna_lr, n_trials=20)

[I 2024-07-12 01:40:53,638] A new study created in memory with name: LogisticRegression
c:\Users\Zylno\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
[I 2024-07-12 01:40:56,749] Trial 0 finished with value: 0.8660550458715597 and parameters: {'penalty': 'l2', 'C': 0.4854676269451529, 'solver': 'saga'}. Best is trial 0 with value: 0.8660550458715597.
[I 2024-07-12 01:40:56,750] Trial 1 finished with value: inf and parameters: {'penalty': 'l1', 'C': 0.634857557664515, 'solver': 'sag'}. Best is trial 1 with value: inf.
[I 2024-07-12 01:40:56,993] Trial 2 finished with value: 0.8782129742962056 and parameters: {'penalty': 'l2', 'C': 0.6890451629020267, 'solver': 'liblinear'}. Best is trial 1 with value: inf.
[I 2024-07-12 01:40:57,368] Trial 3 finished with value: 0.8532277710109623 and parameters: {'penalty': 'l1', 'C': 0.6017896951931353, 'solv

In [71]:
print("Наилучшие значения гиперпараметров {}".format(study_lr.best_params))

# рассчитаем точность для тестовой выборки
model_lr = linear_model.LogisticRegression(**study.best_params,random_state=random_state, )
model_lr.fit(X_train, y_train)

y_test_pred = model_lr.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

Наилучшие значения гиперпараметров {'penalty': 'l1', 'C': 0.634857557664515, 'solver': 'sag'}
f1_score на тестовом наборе: 0.79


c:\Users\Zylno\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


>Значение метрики улучшить не удалось

In [57]:
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 100, 200, 10)
  max_depth = trial.suggest_int('max_depth', 15, 40, 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 3, 10, 1)

  # создаем модель
  model = ensemble.RandomForestClassifier(n_estimators=n_estimators,
                                          max_depth=max_depth,
                                          min_samples_leaf=min_samples_leaf,
                                          random_state=random_state)
  # обучаем модель
  model.fit(X_train, y_train)
  score = metrics.f1_score(y_train, model.predict(X_train))

  return score

# cоздаем объект исследования
study = optuna.create_study(study_name="RandomForestClassifier", direction="maximize")
# ищем лучшую комбинацию гиперпараметров n_trials раз
study.optimize(optuna_rf, n_trials=20)

[I 2024-07-12 01:25:12,529] A new study created in memory with name: RandomForestClassifier
C:\Users\Zylno\AppData\Local\Temp\ipykernel_20772\1962768698.py:3: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
  n_estimators = trial.suggest_int('n_estimators', 100, 200, 10)
C:\Users\Zylno\AppData\Local\Temp\ipykernel_20772\1962768698.py:4: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
  max_depth = trial.suggest_int('max_depth', 15, 40, 1)
C:\Users\Zylno\AppData\Local\Temp\ipykernel_20772\1962768698.py:5: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 3, 10, 1)
[I 2024-07-12 01:25:13,743] Trial 0 finished with value: 0.9593644974029942 and parameters: {'n_estimators': 130, 'max_depth': 31, 'min_samples_leaf

In [59]:
print("Наилучшие значения гиперпараметров {}".format(study.best_params))

# рассчитаем точность для тестовой выборки
model_rf = ensemble.RandomForestClassifier(**study.best_params,random_state=random_state, )
model_rf.fit(X_train, y_train)

y_test_pred = model_rf.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

Наилучшие значения гиперпараметров {'n_estimators': 190, 'max_depth': 25, 'min_samples_leaf': 3}
f1_score на тестовом наборе: 0.83


>Значение метрики улучшить не удалось